# Comparing drought indicators

This notebook uses a small synthetic time series to make the indicator definitions visible without downloading data. It demonstrates anomalies, VCI, TCI, and VHI, then describes how to replace the synthetic inputs with Sentinel-2 and thermal observations.

## Indicator summary

- **NDVI**: vegetation greenness from red and NIR reflectance.
- **NDMI**: vegetation water content from NIR and SWIR reflectance.
- **NDWI**: surface water or wetness signal; definitions vary by application.
- **Anomaly**: departure from a reference distribution.
- **VCI**: NDVI scaled between its historical minimum and maximum.
- **TCI**: land-surface temperature scaled so cooler conditions have higher values.
- **VHI**: weighted combination of VCI and TCI.

VCI and TCI are relative indicators. Their values depend on the baseline years, season, quality masks, and spatial aggregation.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
time = pd.date_range("2024-04-01", periods=12, freq="14D")
ndvi = 0.55 + 0.18 * np.sin(np.linspace(0, np.pi, len(time)))
ndvi[[6, 7, 8]] -= [0.12, 0.20, 0.14]
ndvi += rng.normal(0, 0.015, len(time))
lst = 25 - 4 * np.sin(np.linspace(0, np.pi, len(time)))
lst[[6, 7, 8]] += [2.5, 4.0, 3.0]
lst += rng.normal(0, 0.3, len(time))

data = xr.Dataset({
    "NDVI": ("time", ndvi),
    "LST": ("time", lst),
}, coords={"time": time})
data

The synthetic drought episode occurs around observations 7 to 9: NDVI falls while land-surface temperature rises.

In [ ]:
ndvi_anomaly = data.NDVI - data.NDVI.mean("time")
vci = 100 * (data.NDVI - data.NDVI.min("time")) / (data.NDVI.max("time") - data.NDVI.min("time"))
tci = 100 * (data.LST.max("time") - data.LST) / (data.LST.max("time") - data.LST.min("time"))
vhi = 0.5 * vci + 0.5 * tci

indicators = xr.Dataset({
    "NDVI anomaly": ndvi_anomaly,
    "VCI": vci,
    "TCI": tci,
    "VHI": vhi,
})
indicators

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
data.NDVI.plot(ax=axes[0], marker="o", label="NDVI")
data.LST.plot(ax=axes[0].twinx(), color="firebrick", marker="s", label="LST")
axes[0].set_title("Vegetation and temperature inputs")
indicators["VCI"].plot(ax=axes[1], marker="o", label="VCI")
indicators["TCI"].plot(ax=axes[1], marker="s", label="TCI")
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("condition (%)")
axes[1].legend()
indicators["VHI"].plot(ax=axes[2], marker="o", color="purple")
axes[2].axhline(35, color="firebrick", linestyle="--", label="illustrative threshold")
axes[2].set_ylim(0, 100)
axes[2].set_ylabel("VHI (%)")
axes[2].legend()
plt.tight_layout()

## Extending to real observations

Use the first notebook to produce a quality-masked Sentinel-2 NDVI time series. Obtain land-surface temperature from MODIS or Landsat, then:

1. reproject both datasets to a common grid;
2. aggregate them to the same temporal composite period;
3. calculate each historical baseline from matching seasons;
4. calculate VCI and TCI using the same baseline years; and
5. combine them only after checking spatial coverage and missing observations.

A common formulation is `VHI = alpha * VCI + (1 - alpha) * TCI`, with `alpha=0.5` as a starting point. The weight should be justified for the study region rather than treated as universal. Sentinel-2 alone cannot provide TCI because it has no thermal infrared band.